# Pre-processing

## Introduction
In this tutorial, you'll learn how to use Pandas to perform some basic pre-processing tasks that you may need to carry out to clean your data for the project before you can import it to your database.

Specifically, the tutorial covers:


*   Removal of rows with missing values
*   Replacement of missing values with default values
*   Dectection of entity resolution problems
*   Simple entity resolution
*   Removal of unpaired entities
*   Replacement of categorical variables with indicators
*   Identification of candidate indices
*   Data exportation

## Setup

Run the cell below to import necessary modules.

In [1]:
import numpy as np
import pandas as pd
import os
from google.colab import drive

In [2]:
prefix = '/content/drive'
from google.colab import drive
drive.mount(prefix, force_remount=True)

Mounted at /content/drive


In [ ]:
uber_restaurants_path = '/content/drive/My Drive/cis4500_project/restaurants.csv' # Path to Uber Eats restaurants subset in your drive
uber_menus_path = '/content/drive/My Drive/cis4500_project/restaurant-menus.csv' # Path to Uber Eats menu subset in your drive
yelp_businesses_path = '/content/drive/My Drive/cis4500_project/yelp_academic_dataset_business.json' # Path to Yelp businesses subset in your drive
yelp_reviews_path = '/content/drive/My Drive/cis4500_project/yelp_academic_dataset_review.json' # Path to Yelp reviews subset in your drive

In [ ]:
ur = pd.read_csv(uber_restaurants_path)
um = pd.read_csv(uber_menus_path)

# read Yelp businesses data in chunks because reading in the full thing crashed from using all the RAM

yb_output_path = '/content/drive/My Drive/cis4500_project/yelp_academic_dataset_business.csv'
# first_chunk = True

# for chunk in pd.read_json(yelp_businesses_path, lines=True, chunksize=10000):
#     chunk = chunk[chunk['categories'].fillna('').str.contains('Restaurants', case=False)] # Keeps restaurant businesses
#     chunk.to_csv(yb_output_path, mode='w' if first_chunk else 'a',
#                  header=first_chunk, index=False)
#     first_chunk = False

yb = pd.read_csv(yb_output_path)

# read Yelp review data in chunks because reading in the full thing crashed from using all the RAM
yr_output_path = '/content/drive/My Drive/cis4500_project/yelp_academic_dataset_review.csv'

restaurants_set = set(yb['business_id'])
first_chunk = True

for chunk in pd.read_json(yelp_reviews_path, lines=True, chunksize=10000):
    chunk = chunk[chunk['business_id'].isin(restaurants_set)] # Keeps restaurant businesses
    chunk.to_csv(yr_output_path, mode='w' if first_chunk else 'a',
                 header=first_chunk, index=False)
    first_chunk = False

yr = pd.read_csv(yr_output_path)

yr.head()

,review_id,user_id,business_id,stars,useful,funny,cool,text,date
0,KU_O5udG6zpxOg-VcAEodg,mh_-eMZ6K5RLWhZyISBhwA,XQfwVwDr-v0ZS3_CbbE5Xw,3,0,0,0,"If you decide to eat here, just be aware it is...",2018-07-07 22:09:11
1,saUsX_uimxRlCVr67Z4Jig,8g_iMtfSiwikVnbP2etR0A,YjUWPpI6HXG530lwP-fb2A,3,0,0,0,Family diner. Had the buffet. Eclectic assortm...,2014-02-05 20:30:30
2,AqPFMleE6RsU23_auESxiA,_7bHUi9Uuf5__HHc_Q8guQ,kxX2SOes4o-D3ZQBkiMRfA,5,1,0,1,"Wow! Yummy, different, delicious. Our favo...",2015-01-04 00:01:03
3,Sx8TMOWLNuJBWer-0pcmoA,bcjbaE6dDog4jkNY91ncLQ,e4Vwtrqf-wpJfwesgvdgxQ,4,1,0,1,Cute interior and owner (?) gave us tour of up...,2017-01-14 20:54:15
4,JrIxlS1TzJ-iCu79ul40cQ,eUta8W_HdHMXPzLBBZhL1A,04UD14gamNjLY0IDYVhHJg,1,1,2,1,I am a long term frequent customer of this est...,2015-09-23 23:10:31


First, we generate a dataframe where each element indicates whether that element was missing in the original dataframe.

In [ ]:
ur.dtypes
um.dtypes
yb.dtypes
yr.dtypes
print(ur.shape, um.shape, yb.shape, yr.shape)
um.isnull().any()
um[['name']].isna().sum()

(63469, 11) (5117217, 5) (52268, 14) (4724471, 9)


,0
name,4


In [ ]:
ur['score'] = ur['score'].fillna(ur['score'].mean()) # Fill null scores with the average to avoid errors when computing aggregate statistics while still maintaining the center
ur['ratings'] = ur['ratings'].fillna(0) # Null values have zero ratings
ur['category'] = ur['category'].fillna('unknown')
ur['price_range'] = ur['price_range'].fillna('unknown')
um = um.dropna(subset=['name']) # Drop 4 rows with null names

In [5]:
# Remove USD from price and convert the data type
um["price"] = (um["price"].astype(str).str.replace(r"[^\d.]", "", regex=True))  # Remove everything except digits and dot
um["price"] = pd.to_numeric(um["price"], errors="coerce")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Remove rows, replace with defaults, or ignore?
None of these options is strictly better than the others. Depending on your application and what exactly is missing from the data, any choice could be acceptable.

This doesn't mean you can make the decision thoughtlessly. In context, certain choices can be misguided. For example, you can't ignore missing values in a column you plan to use as a table index, and you can't throw out all rows with missing values if 95% of your rows are incomplete.

So for the project, think carefully about which approach is appropriate for your application and data. Feel free to have a discussion with your project mentor. We expect you to justify whatever decision you make in your project report.

## Entity Resolution

Now that we've handled missing values in both datasets, we turn our attention to performing entity resolution on the entities common to both. In this case, those common entities are countries.

To perform entity resolution, we will:
1. Determine whether both datasets use the same names to refer to all countries
2. Edit the names in one dataset to match the other, if necessary

In the general case, you may also need to detect when datasets refer to different entities using the same name/ID and disambiguate these references. This may happen when handling datasets that contain multiple people with the same name, for example. But we don't need to worry about it here because the names of countries are well-known and distict.


### Detect Inconsistent Names
Let's compile a list of all country names in both datasets, then inspect it for repetitions.

First, we extract the unique names of countries from both datasets.

In [ ]:
import re

yb["name_norm"] = yb["name"].str.lower().str.strip().str.replace(r"[^\w\s]", "", regex=True) # remove punctuation, but keep original name to display
ur["name_norm"] = ur["name"].str.lower().str.strip().str.replace(r"[^\w\s]", "", regex=True)

# Split full address into address, city, and state

valid_states = {
    "al","ak","az","ar","ca","co","ct","de","fl","ga","hi","id","il","in","ia","ks","ky","la",
    "me","md","ma","mi","mn","ms","mo","mt","ne","nv","nh","nj","nm","ny","nc","nd","oh","ok",
    "or","pa","ri","sc","sd","tn","tx","ut","vt","va","wa","wv","wi","wy","dc","pr"
}

def extract_address_city_state(full_address):
    if pd.isna(full_address):
        return pd.Series([None, None, None])

    parts = [p.strip().lower() for p in str(full_address).split(",")]
    parts = [p for p in parts if p != ""]

    state_idx = None
    for i in range(len(parts) - 1, -1, -1):
        if parts[i] in valid_states:
            state_idx = i
            break

    if state_idx is None:
        return pd.Series([None, None, None])

    state = parts[state_idx]
    city = parts[state_idx - 1] if state_idx - 1 >= 0 else None
    address_parts = parts[:state_idx - 1]
    address = ", ".join(address_parts) if address_parts else None

    return pd.Series([address, city, state])

ur[["extracted_address", "city_extracted", "state_extracted"]] = ur["full_address"].apply(extract_address_city_state)


yb["city_norm"] = yb["city"].str.lower().str.strip()
yb["state_norm"] = yb["state"].str.lower().str.strip()

ur["city_norm"] = ur["city_extracted"].str.lower().str.strip()
ur["state_norm"] = ur["state_extracted"].str.lower().str.strip()
ur.head(100)

,id,position,name,score,ratings,category,price_range,full_address,zip_code,lat,lng,name_norm,extracted_address,city_extracted,state_extracted,city_norm,state_norm
0,1,19,PJ Fresh (224 Daniel Payne Drive),4.551431,0.0,"Burgers, American, Sandwiches",$,"224 Daniel Payne Drive, Birmingham, AL, 35207",35207,33.562365,-86.830703,pj fresh 224 daniel payne drive,224 daniel payne drive,birmingham,al,birmingham,al
1,2,9,J' ti`'z Smoothie-N-Coffee Bar,4.551431,0.0,"Coffee and Tea, Breakfast and Brunch, Bubble Tea",unknown,"1521 Pinson Valley Parkway, Birmingham, AL, 35217",35217,33.583640,-86.773330,j tiz smoothiencoffee bar,1521 pinson valley parkway,birmingham,al,birmingham,al
2,3,6,Philly Fresh Cheesesteaks (541-B Graymont Ave),4.551431,0.0,"American, Cheesesteak, Sandwiches, Alcohol",$,"541-B Graymont Ave, Birmingham, AL, 35204",35204,33.509800,-86.854640,philly fresh cheesesteaks 541b graymont ave,541-b graymont ave,birmingham,al,birmingham,al
3,4,17,Papa Murphy's (1580 Montgomery Highway),4.551431,0.0,Pizza,$,"1580 Montgomery Highway, Hoover, AL, 35226",35226,33.404439,-86.806614,papa murphys 1580 montgomery highway,1580 montgomery highway,hoover,al,hoover,al
4,5,162,Nelson Brothers Cafe (17th St N),4.700000,22.0,"Breakfast and Brunch, Burgers, Sandwiches",unknown,"314 17th St N, Birmingham, AL, 35203",35203,33.514730,-86.811700,nelson brothers cafe 17th st n,314 17th st n,birmingham,al,birmingham,al
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,96,147,Zulas,4.551431,0.0,"Burgers, American, Sandwiches",$,"2125 Highland Ave, Birmingham, AL, 35205",35205,33.499634,-86.792458,zulas,2125 highland ave,birmingham,al,birmingham,al
96,97,144,The Southern Kitchen &amp; Bar,4.551431,0.0,"BBQ, American, Burgers",$,"2301 Richard Arrington Jr Blvd N, Birmingham, ...",35203,33.526170,-86.807780,the southern kitchen amp bar,2301 richard arrington jr blvd n,birmingham,al,birmingham,al
97,98,142,Potatoe Potatohz Perfic Pizza,4.551431,0.0,"Pizza, American, Wings",unknown,"707 Richard Arrington Jr Blvd S, Birmingham, A...",35233-2105,33.506550,-86.798290,potatoe potatohz perfic pizza,707 richard arrington jr blvd s,birmingham,al,birmingham,al
98,99,141,Martys,4.551431,0.0,"American, Burgers, Exclusive to Eats",$$,"1813 10th Ct S, Birmingham, AL, 35205",35205,33.500181,-86.799315,martys,1813 10th ct s,birmingham,al,birmingham,al


Now, we combine these into one set.

In [ ]:
yelp_states = yb['state'].unique()
uber_states = ur['state_extracted'].unique()
states = set(yelp_states.tolist() + uber_states.tolist())
ur = ur.dropna(subset=["state_extracted"]) # drop 624 rows with null states
ur[['state_extracted']].isna().sum()

,0
state_extracted,0


## Find an Index

Before ingesting our data into the database, we need to find a unique index for each table. Let's do this for the life expectancy dataset.

### Single Column Index
The fastest way to determine whether any single column is unique is to check whether the number of unique values in the candidate column equals the number of elements. For example, let's find out whether `country` could be the index.  

In [ ]:
len(ur["id"].unique()) == len(ur["id"])
len(yb["business_id"].unique()) == len(yb["business_id"])
len(yr["review_id"].unique()) == len(yr["review_id"])

True

### Multi-Column Index

When there's not an individual column that can act as an index, we search for combinations of columns that can make a unique index when combined.

To check a candidate set of columns:
1. Call `DataFrame.groupby()` on the list of candidate columns. *This creates a group of rows for each unique combination of candidate olumn values that appears in the Dataframe*
2. Call `GroupBy.size()` on the resulting grouped dataframe. *This counts the number of rows in each group*
3. Check whether every group has exactly 1 row.

We use this procedure below to check whether `country` and `year` can function as a joint index.

In [103]:
um = um.drop_duplicates().reset_index(drop=True) # create an index because every column is needed to make a unique index
um["menu_item_id"] = um.index


,restaurant_id,category,name,description,price,menu_item_id
0,1,Extra Large Pizza,Extra Large Meat Lovers,Whole pie.,15.99 USD,0
1,1,Extra Large Pizza,Extra Large Supreme,Whole pie.,15.99 USD,1
2,1,Extra Large Pizza,Extra Large Pepperoni,Whole pie.,14.99 USD,2
3,1,Extra Large Pizza,Extra Large BBQ Chicken &amp; Bacon,Whole Pie,15.99 USD,3
4,1,Extra Large Pizza,Extra Large 5 Cheese,Whole pie.,14.99 USD,4


Great! This means no country-year pair appears more than once in the life expectancy table, so we can use `country` and `year` in combination as the index.

For the census dataset, you'd need every column to create a unique index (Check this for yourself). So we'll just plan to use the arbitrary, unique integers `census.index` as our table index.

## Export Data
After cleaning our datasets, resolving entity resolution problems, and choosing indices, we're ready to ingest our the data into our database.

In the past, students have found using Python for data ingestion slow and frustrating, so we won't populate the database here. Instead, we'll export both datasets and the country codes to CSVs. For now we use DataGrip for data ingestion, here are some reference for the [DataGrip DataSource](https://www.jetbrains.com/help/datagrip/managing-data-sources.html)


Next, we use `DataFrame.to_csv` to write each dataset to a CSV file with a descriptive name.

*For `le` and `country_codes`, we set `index` to `False` because the indices of the `DataFrames` are meaningless integers that we don't need in our tables. For `census`, we rename the index and include it in the output because we decided  to use it as our table index, even though it's arbitrary.*

In [ ]:
yb.to_csv("yelp_businesses.csv", index=False)
yr.to_csv("yelp_reviews.csv", index=False)
ur.to_csv("uber_restaurants.csv", index=False)
um.to_csv("uber_menus.csv", index=False)

Finally, we download these files to our local machine, so we can put them into MySQL Workbench later.

In [ ]:
from google.colab import files
files.download("yelp_businesses.csv")
files.download("yelp_reviews.csv")
files.download("uber_restaurants.csv")
files.download("uber_menus.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Exercises
Check out [these exercises](https://drive.google.com/open?id=1kjLaYC_KJUlltm-iAzgF5VmHgb7Uer0z) to practice the processing techniques you learned above!